# CSE4078 Spring 2026 — Group 5 — Optimized Fine-Tuning
## QLoRA Fine-Tuning of Qwen3-4B — Improved Hyperparameters

**Members:** Yiğitcan Değişme, Emir Devran, Kerem Kolay, Yunus Emre Gül, Ömer Faruk Sevinç  
**Dataset:** alibayram/doktorsitesi  
**Model:** Qwen/Qwen3-4B  
**Platform:** Google Colab Pro — NVIDIA A100 40GB GPU

---
### Değişiklikler (Deadline 2'ye göre)

| Parametre | Deadline 2 | Bu Notebook |
|-----------|-----------|-------------|
| Epoch | 1 | **3** |
| LoRA rank (r) | 16 | **32** |
| LoRA alpha | 16 | **64** |
| LoRA dropout | 0.0 | **0.05** |
| Learning rate | 2e-4 | **1e-4** |
| LR scheduler | linear | **cosine** |
| Warmup | 5 steps | **0.03 ratio** |
| Training data | 15,000 | **39,612 (tamamı)** |

---
### Notebook Structure
1. Data Preprocessing
2. Model Setup — Qwen3-4B + QLoRA (r=32)
3. SFT Data Preparation — Tüm 39,612 örnek
4. Optimized Fine-Tuning — 3 epoch
5. Save Model
6. Post-SFT Inference
7. Evaluation — 8 Metrik

---
## 1. Data Preprocessing

In [1]:
!pip install datasets pandas scikit-learn huggingface_hub

In [2]:
from huggingface_hub import notebook_login
notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [4]:
from datasets import load_dataset
import pandas as pd
from sklearn.model_selection import train_test_split
import json

print("Veri kümesi indiriliyor...")

dataset = load_dataset("alibayram/doktorsitesi")
df = pd.DataFrame(dataset['train'])

hedef_branslar = [
    "beyin-ve-sinir-cerrahisi",
    "uroloji",
    "ortopedi-ve-travmatoloji",
    "dahiliye-ve-ic-hastaliklari",
    "genel-cerrahi",
    "kulak-burun-bogaz-hastaliklari",
    "fiziksel-tip-ve-rehabilitasyon",
    "kardiyoloji",
    "kalp-damar-cerrahisi"
]

df_filtered = df[df['doctor_speciality'].isin(hedef_branslar)].copy()

# NaN satırları temizle — boş soru/cevap training'i bozar
before_na = len(df_filtered)
df_filtered = df_filtered.dropna(subset=['question_content', 'question_answer'])
print(f"NaN temizlendi: {before_na - len(df_filtered)} satır silindi")

# Deduplikasyon
before = len(df_filtered)
df_filtered = df_filtered.drop_duplicates(subset=['question_content'])
print(f"Tekrar silindi: {before - len(df_filtered)}")

# Stratified split (seed=42)
train_df, test_df = train_test_split(
    df_filtered,
    test_size=0.1,
    stratify=df_filtered['doctor_speciality'],
    random_state=42
)

print(f"Train: {len(train_df)} | Test: {len(test_df)}")

instruction = "Aşağıdaki tıbbi soruyu Türkçe ve dikkatli biçimde yanıtla."

with open("sft_train.jsonl", "w", encoding="utf-8") as f:
    for _, row in train_df.iterrows():
        record = {
            "instruction": instruction,
            "input": row["question_content"],
            "output": row["question_answer"],
            "doctor_speciality": row["doctor_speciality"]
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

with open("benchmark_test.jsonl", "w", encoding="utf-8") as f:
    for _, row in test_df.iterrows():
        record = {
            "instruction": instruction,
            "input": row["question_content"],
            "output": row["question_answer"],
            "doctor_speciality": row["doctor_speciality"]
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

train_q = set(train_df['question_content'].tolist())
test_q  = set(test_df['question_content'].tolist())
print(f"Train-Test overlap: {len(train_q & test_q)} (0 olmalı)")
print("Dosyalar kaydedildi!")


Veri kümesi indiriliyor...


train-00000-of-00001.parquet:   0%|          | 0.00/66.5M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/16.7M [00:00<?, ?B/s]

(…)tle_spec_combined-00000-of-00001.parquet:   0%|          | 0.00/83.0M [00:00<?, ?B/s]

balanced_2_title-00000-of-00001.parquet:   0%|          | 0.00/3.39M [00:00<?, ?B/s]

...

Generating test split:   0%|          | 0/37527 [00:00<?, ? examples/s]

Generating title_spec_combined split:   0%|          | 0/187632 [00:00<?, ? examples/s]

Generating balanced_2_title split:   0%|          | 0/8000 [00:00<?, ? examples/s]

Generating balanced_2_title_test split:   0%|          | 0/800 [00:00<?, ? examples/s]

NaN temizlendi: 0 satır silindi
Tekrar silindi: 4824
Train: 36412 | Test: 4046
Train-Test overlap: 0 (0 olmalı)
Dosyalar kaydedildi!


---
## 2. Model Setup — Qwen3-4B + QLoRA (r=32, alpha=64)

In [5]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install --no-deps peft accelerate bitsandbytes -q

from trl import SFTConfig
print(f"TRL SFTConfig OK — trl version: {__import__('trl').__version__}")


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 141.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 67.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 128.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 124.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 27.5 MB/s eta 0:00:00


TRL SFTConfig OK — trl version: 0.24.0


In [6]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen3-4B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    lora_alpha = 64,
    lora_dropout = 0.05,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
print("Qwen3-4B modeli QLoRA (r=32) için hazırlandı!")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/usr/local/lib/python3.12/dist-packages/unsloth/__init__.py:153: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.7: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/3.55G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

...

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/4.67k [00:00<?, ?B/s]

unsloth/qwen3-4b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.6.7 patched 36 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


Qwen3-4B modeli QLoRA (r=32) için hazırlandı!


---
## 3. SFT Data Preparation — Tüm 39,612 Örnek

In [7]:
import pandas as pd
from datasets import Dataset
import json

assert 'tokenizer' in dir(), "tokenizer bulunamadı — önce Cell 7'yi çalıştır!"

with open("sft_train.jsonl", encoding="utf-8") as fh:
    records = [json.loads(l) for l in fh]

train_df = pd.DataFrame([{
    "question_content": r["input"],
    "question_answer":  r["output"],
    "doctor_speciality": r["doctor_speciality"]
} for r in records])
train_df = train_df.reset_index(drop=True)

print(f"Eğitim verisi: {len(train_df)} örnek")
print(train_df['doctor_speciality'].value_counts())

PROMPT_PREFIX = (
    "Aşağıda bir hastanın tıbbi sorusu ve bir uzman doktorun buna verdiği yanıt yer almaktadır.\n\n"
    "### Soru:\n"
)
PROMPT_MID = "\n\n### Cevap:\n"
EOS_TOKEN = tokenizer.eos_token or ""

def formatting_prompts_func(examples):
    questions = examples["question_content"]
    answers   = examples["question_answer"]
    texts = []
    for q, a in zip(questions, answers):
        text = PROMPT_PREFIX + str(q) + PROMPT_MID + str(a) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}

sft_dataset = Dataset.from_pandas(train_df)
sft_dataset = sft_dataset.map(formatting_prompts_func, batched=True)
print(f"Veri SFT formatına dönüştürüldü: {len(sft_dataset)} örnek")


Eğitim verisi: 36412 örnek
doctor_speciality
beyin-ve-sinir-cerrahisi          9310
uroloji                           7609
ortopedi-ve-travmatoloji          4979
genel-cerrahi                     4423
kulak-burun-bogaz-hastaliklari    3478
fiziksel-tip-ve-rehabilitasyon    3238
kardiyoloji                       1751
kalp-damar-cerrahisi               929
dahiliye-ve-ic-hastaliklari        695
Name: count, dtype: int64


Map:   0%|          | 0/36412 [00:00<?, ? examples/s]

Veri SFT formatına dönüştürüldü: 36412 örnek


---
## 4. Optimized Fine-Tuning — 3 Epoch, Cosine LR

In [9]:
from trl import SFTTrainer, SFTConfig
import torch

assert 'sft_dataset' in dir(), "sft_dataset bulunamadı — önce Cell 9'u çalıştır!"
assert 'model' in dir(), "model bulunamadı — önce Cell 7'yi çalıştır!"

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = sft_dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        max_length = 1024,
        dataset_num_proc = 1,
        packing = False,
        padding_free = False,
        loss_type = "nll",
        report_to = "none",

        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,
        num_train_epochs = 3,
        warmup_steps = 200,
        learning_rate = 1e-4,
        lr_scheduler_type = "cosine",
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 50,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        save_strategy = "epoch",
        save_total_limit = 1,          # Disk tasarrufu: sadece son checkpoint
        seed = 3407,
        output_dir = "outputs_optimized",
    ),
)

torch.cuda.empty_cache()
print("Optimize edilmiş eğitim başlatılıyor...")
print("39,612 örnek | 3 epoch | r=32 | cosine LR")
print("Tahmini süre: 3-4 saat (A100)")
trainer_stats = trainer.train()
print("Eğitim tamamlandı!")


Unsloth: Tokenizing ["text"]:   0%|          | 0/36412 [00:00<?, ? examples/s]

Optimize edilmiş eğitim başlatılıyor...
39,612 örnek | 3 epoch | r=32 | cosine LR
Tahmini süre: 3-4 saat (A100)


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 36,412 | Num Epochs = 3 | Total steps = 6,828
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 66,060,288 of 4,088,528,384 (1.62% trained)


Step,Training Loss
50,1.841771
100,1.774398
150,1.786730
200,1.727799
250,1.736029
300,1.700885
350,1.717347
400,1.690596
450,1.666313
500,1.662493


Unsloth: Restored added_tokens_decoder metadata in outputs_optimized/checkpoint-2276/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs_optimized/checkpoint-4552/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs_optimized/checkpoint-6828/tokenizer_config.json.


Eğitim tamamlandı!


---
## 5. Save Optimized Model

In [10]:
model.save_pretrained("qwen3_doc_lora_optimized")
tokenizer.save_pretrained("qwen3_doc_lora_optimized")
print("Model 'qwen3_doc_lora_optimized' olarak kaydedildi!")

# Google Drive'a yedekle
from google.colab import drive
import shutil, os

drive.mount('/content/drive')

dest = '/content/drive/MyDrive/qwen3_doc_lora_optimized'
if os.path.exists(dest):
    shutil.rmtree(dest)   # Önceki kopya varsa sil
shutil.copytree('qwen3_doc_lora_optimized', dest)
print("Drive'a yedeklendi:", dest)


Unsloth: Restored added_tokens_decoder metadata in qwen3_doc_lora_optimized/tokenizer_config.json.


Model 'qwen3_doc_lora_optimized' olarak kaydedildi!
Mounted at /content/drive
Drive'a yedeklendi: /content/drive/MyDrive/qwen3_doc_lora_optimized


---
## 6. Post-SFT Inference on benchmark_test.jsonl

In [13]:
from unsloth import FastLanguageModel
import pandas as pd
import torch
import json
import os

if 'model' not in globals() or not hasattr(model, 'generate'):
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="qwen3_doc_lora_optimized",
        max_seq_length=2048,
        dtype=None,
        load_in_4bit=True,
    )

FastLanguageModel.for_inference(model)
model.generation_config.max_length = None

_pad_id = tokenizer.eos_token_id or tokenizer.pad_token_id or 0

with open("benchmark_test.jsonl", encoding="utf-8") as fh:
    records = [json.loads(l) for l in fh]
print(f"Test seti: {len(records)} satır")

BATCH_SIZE = 16

def generate_batch(questions):
    prompts = []
    for question in questions:
        messages = [
            {"role": "system", "content": "Aşağıdaki tıbbi soruyu Türkçe ve dikkatli biçimde yanıtla."},
            {"role": "user",   "content": question},
        ]
        try:
            prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
        except TypeError:
            prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        prompts.append(prompt)

    inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=2048).to("cuda")
    input_length = inputs["input_ids"].shape[1]
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=64, do_sample=False, pad_token_id=_pad_id)
    return [tokenizer.decode(out[input_length:], skip_special_tokens=True) for out in outputs]

out_file = "qwen3_optimized_predictions.csv"

if os.path.exists(out_file):
    done_df = pd.read_csv(out_file)
    done_indices = set(done_df["index"].astype(int).tolist())
    rows = done_df.to_dict("records")
    print(f"Kaldığı yerden devam: {len(done_indices)} tamamlanmış")
else:
    done_indices = set()
    rows = []

todo_recs    = [r for i, r in enumerate(records) if i not in done_indices][:1500]
todo_indices = [i for i in range(len(records)) if i not in done_indices][:1500]
print(f"Üretilecek: {len(todo_recs)} örnek")

for batch_start in range(0, len(todo_recs), BATCH_SIZE):
    batch_recs = todo_recs[batch_start : batch_start + BATCH_SIZE]
    batch_idx  = todo_indices[batch_start : batch_start + BATCH_SIZE]
    try:
        preds = generate_batch([r["input"] for r in batch_recs])
    except Exception as e:
        print(f"[UYARI] Batch {batch_start} hata: {e}")
        torch.cuda.empty_cache()
        preds = ["ERROR"] * len(batch_recs)
    for idx, rec, pred in zip(batch_idx, batch_recs, preds):
        rows.append({"index": idx, "doctor_speciality": rec["doctor_speciality"],
                     "question": rec["input"], "reference": rec["output"], "prediction": pred})
    done_count = batch_start + len(batch_recs)
    if done_count % 200 == 0 or done_count >= len(todo_recs):
        pd.DataFrame(rows).to_csv(out_file, index=False)
        print(f"{len(done_indices) + done_count}/{len(records)} tamamlandı...")

pd.DataFrame(rows).to_csv(out_file, index=False)
print("Inference tamamlandı! -> qwen3_optimized_predictions.csv")

pd.DataFrame(rows).to_csv(out_file, index=False)
print("Inference tamamlandı! -> qwen3_optimized_predictions.csv")


Test seti: 4046 satır
Kaldığı yerden devam: 650 tamamlanmış
Üretilecek: 1500 örnek


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

1050/4046 tamamlandı...
1450/4046 tamamlandı...
1850/4046 tamamlandı...
2150/4046 tamamlandı...
Inference tamamlandı! -> qwen3_optimized_predictions.csv
Inference tamamlandı! -> qwen3_optimized_predictions.csv


---
## 7. Evaluation — All 8 Metrics

In [14]:
!pip install rouge-score bert-score sentence-transformers -q

from rouge_score import rouge_scorer
from bert_score import score as bert_score
from sentence_transformers import SentenceTransformer, util
import pandas as pd
import numpy as np
import unicodedata, re

df = pd.read_csv("qwen3_optimized_predictions.csv").head(1500)
print(f"Toplam satır: {len(df)}")

references  = df['reference'].fillna("").tolist()
predictions = df['prediction'].fillna("").tolist()

# Normalizasyon
def normalize(text):
    text = unicodedata.normalize("NFKD", str(text))
    text = text.casefold()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Exact Match
df['exact_match'] = [int(normalize(r) == normalize(p))
                     for r, p in zip(references, predictions)]

# Token F1
def token_f1(ref, pred):
    r = normalize(ref).split()
    p = normalize(pred).split()
    if not r or not p:
        return 0.0
    common = set(r) & set(p)
    if not common:
        return 0.0
    prec = len(common) / len(p)
    rec  = len(common) / len(r)
    return 2 * prec * rec / (prec + rec)

df['token_f1'] = [token_f1(r, p) for r, p in zip(references, predictions)]

# ROUGE
scorer = rouge_scorer.RougeScorer(['rouge1','rouge2','rougeL'], use_stemmer=False)
r1, r2, rL = [], [], []
for ref, pred in zip(references, predictions):
    s = scorer.score(ref, pred)
    r1.append(s['rouge1'].fmeasure)
    r2.append(s['rouge2'].fmeasure)
    rL.append(s['rougeL'].fmeasure)
df['rouge1_f1'] = r1
df['rouge2_f1'] = r2
df['rougeL_f1'] = rL

# MiniLM
print("MiniLM hesaplanıyor...")
sim_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
emb_pred = sim_model.encode(predictions, batch_size=64, convert_to_tensor=True)
emb_ref  = sim_model.encode(references,  batch_size=64, convert_to_tensor=True)
sims = util.cos_sim(emb_pred, emb_ref).diagonal().cpu().numpy()
df['minilm_sim'] = sims

# BERTScore
print("BERTScore hesaplanıyor...")
P, R, F1 = bert_score(predictions, references,
                      lang='tr', model_type='xlm-roberta-base',
                      device='cuda', batch_size=32, verbose=True)
df['bertscore_f1'] = F1.numpy()

# Length
df['gen_tokens'] = [len(str(p).split()) for p in predictions]

df.to_csv("qwen3_optimized_metrics.csv", index=False)

# Karşılaştırma tablosu
deadline2 = {
    'Exact Match':  0.0013,
    'Token F1':     0.0811,
    'ROUGE-1 F1':   0.1302,
    'ROUGE-2 F1':   0.0326,
    'ROUGE-L F1':   0.0978,
    'MiniLM Sim.':  0.4121,
    'BERTScore F1': 0.8283,
    'Avg Tokens':   31.6,
}
results = {
    'Exact Match':  df['exact_match'].mean(),
    'Token F1':     df['token_f1'].mean(),
    'ROUGE-1 F1':   df['rouge1_f1'].mean(),
    'ROUGE-2 F1':   df['rouge2_f1'].mean(),
    'ROUGE-L F1':   df['rougeL_f1'].mean(),
    'MiniLM Sim.':  df['minilm_sim'].mean(),
    'BERTScore F1': df['bertscore_f1'].mean(),
    'Avg Tokens':   df['gen_tokens'].mean(),
}

print("\n" + "="*65)
print(f"{'METRİK':<20} {'DEADLINE 2':>12} {'OPTİMİZE':>12} {'DELTA':>10}")
print("="*65)
for k in deadline2:
    d2, opt = deadline2[k], results[k]
    delta = opt - d2
    print(f"{k:<20} {d2:>12.4f} {opt:>12.4f} {'↑' if delta>0 else '↓'}{abs(delta):>8.4f}")

# Alan bazında
print("\n=== ALAN BAZINDA TOKEN F1 ===")
deadline2_alan = {
    'beyin-ve-sinir-cerrahisi':       0.0757,
    'uroloji':                        0.0905,
    'ortopedi-ve-travmatoloji':       0.0796,
    'genel-cerrahi':                  0.0772,
    'kulak-burun-bogaz-hastaliklari': 0.0738,
    'fiziksel-tip-ve-rehabilitasyon': 0.0933,
    'kardiyoloji':                    0.0824,
    'kalp-damar-cerrahisi':           0.0756,
    'dahiliye-ve-ic-hastaliklari':    0.0706,
}
alan = df.groupby('doctor_speciality')['token_f1'].mean().round(4)
print(f"{'Alan':<40} {'Deadline 2':>12} {'Optimize':>10} {'Delta':>8}")
print("-"*74)
for alan_adi, opt_val in alan.items():
    d2_val = deadline2_alan.get(alan_adi, 0)
    delta = opt_val - d2_val
    print(f"{alan_adi:<40} {d2_val:>12.4f} {opt_val:>10.4f} {'↑' if delta>0 else '↓'}{abs(delta):>6.4f}")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.9 MB/s eta 0:00:00
Toplam satır: 1500
MiniLM hesaplanıyor...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/87 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/47 [00:00<?, ?it/s]

done in 3.81 seconds, 394.11 sentences/sec

METRİK                 DEADLINE 2     OPTİMİZE      DELTA
Exact Match                0.0013       0.0033 ↑  0.0020
Token F1                   0.0811       0.1006 ↑  0.0195
ROUGE-1 F1                 0.1302       0.1487 ↑  0.0185
ROUGE-2 F1                 0.0326       0.0471 ↑  0.0145
ROUGE-L F1                 0.0978       0.1183 ↑  0.0205
MiniLM Sim.                0.4121       0.4504 ↑  0.0383
BERTScore F1               0.8283       0.8363 ↑  0.0080
Avg Tokens                31.6000      25.6880 ↓  5.9120

=== ALAN BAZINDA TOKEN F1 ===
Alan                                       Deadline 2   Optimize    Delta
--------------------------------------------------------------------------
beyin-ve-sinir-cerrahisi                       0.0757     0.1131 ↑0.0374
dahiliye-ve-ic-hastaliklari                    0.0706     0.0601 ↓0.0105
fiziksel-tip-ve-rehabilitasyon                 0.0933     0.1117 ↑0.0184
genel-cerrahi                              